# Evaluation of the road-rules chatbot

Open the **Ollama** app, then run the cells from top to bottom.
First run with `LIMIT = 6` (quick test). When it works, set `LIMIT = None` and run again.

## Install packages

In [1]:
%pip install -q chromadb sentence-transformers ollama rouge-score bert-score pandas

Note: you may need to restart the kernel to use updated packages.


## Settings

In [2]:
RUN_NAME = "baseline"   # "baseline" for the first run, "tuned" for the second
SPLIT = "test"          # "test", "tune" or "all"
LIMIT = 6               # 6 = quick test, None = all questions
MODEL = "llama3.2:3b"
THRESHOLD = 0.6         # DKT accuracy: how close an answer must be (0 to 1)

## Setup

In [3]:
import os, re, json, glob, time, random
import pandas as pd

GOLDEN_FILE = "data/golden/golden_set_v3.jsonl"

def find_project_folder():
    folder = os.getcwd()
    while folder != os.path.dirname(folder):          # this folder and the ones above it
        if os.path.exists(os.path.join(folder, GOLDEN_FILE)):
            return folder
        folder = os.path.dirname(folder)
    for depth in range(4):                            # then look inside your home folder
        for path in glob.glob(os.path.expanduser("~/" + "*/" * depth + "road-rules-rag")):
            if os.path.exists(os.path.join(path, GOLDEN_FILE)):
                return path
    raise FileNotFoundError("Save this notebook inside the road-rules-rag folder and run it again.")

os.chdir(find_project_folder())
RESULTS_DIR = os.path.expanduser("~/road_rules_results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Project folder:", os.getcwd())

Project folder: /Users/vineetbangera/Downloads/road-rules-rag


## Load the questions

In [4]:
golden = [json.loads(line) for line in open(GOLDEN_FILE, encoding="utf-8")]
questions = [q for q in golden if SPLIT == "all" or q["split"] == SPLIT]

if LIMIT:   # quick test: a mix of in-scope and out-of-scope questions
    n_out = max(1, LIMIT // 3)
    inside = [q for q in questions if q["in_kb"]]
    outside = [q for q in questions if not q["in_kb"]]
    questions = inside[:LIMIT - n_out] + outside[:n_out]

print("Questions to run:", len(questions))

Questions to run: 6


## Open the handbook database

In [5]:
import chromadb
from chromadb.utils import embedding_functions

ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path=os.path.expanduser("~/road_rules_chroma"))
col = client.get_collection("handbook", embedding_function=ef)   # built in notebook 03
print("Chunks in database:", col.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunks in database: 348


## Ask the chatbot every question

In [6]:
import ollama

ANSWERS_FILE = f"{RESULTS_DIR}/answers_{RUN_NAME}.json"
generated = json.load(open(ANSWERS_FILE, encoding="utf-8")) if os.path.exists(ANSWERS_FILE) else {}

def get_cited_pages(text):
    pages = []
    for match in re.findall(r"[Pp]ages?\s*((?:\d+(?:\s*(?:,|and|&)\s*)?)+)", text):
        pages += [int(n) for n in re.findall(r"\d+", match)]
    return sorted(set(pages))

for n, q in enumerate(questions, 1):
    if q["id"] in generated:
        continue                                       # already answered in an earlier run

    result = col.query(query_texts=[q["question"]], n_results=5)
    chunks = result["documents"][0]
    pages = [m["page"] for m in result["metadatas"][0]]
    context = "\n\n".join(f"[Page {p}]\n{c}" for c, p in zip(chunks, pages))

    prompt = f"""You are a NSW road rules assistant.

Answer the question ONLY using the NSW Road User Handbook information provided below.

Do not use outside knowledge.
Do not invent road rules.

If the answer cannot be found in the provided context,
say exactly: "I cannot find this information in the provided NSW Road User Handbook."

Include the relevant handbook page in your answer.

HANDBOOK CONTEXT:
{context}

QUESTION:
{q["question"]}
"""
    start = time.time()
    answer = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}],
                         options={"temperature": 0, "num_predict": 200})["message"]["content"]
    seconds = round(time.time() - start, 1)

    generated[q["id"]] = {
        "question": q["question"],
        "in_scope": q["in_kb"],
        "answer": answer,
        "reference": q.get("reference_answer"),
        "dkt_answer": q.get("answer"),
        "gold_page": q.get("gold_page"),
        "refused": "cannot find this information" in answer.lower(),
        "cited_pages": get_cited_pages(answer),
        "retrieved_text": chunks,
        "seconds": seconds,
    }
    json.dump(generated, open(ANSWERS_FILE, "w", encoding="utf-8"), indent=1)   # saves progress
    print(f"[{n}/{len(questions)}] {q['id']}  {seconds}s  refused={generated[q['id']]['refused']}")

generated = {q["id"]: generated[q["id"]] for q in questions}
in_scope = {k: r for k, r in generated.items() if r["in_scope"]}        # should be answered
out_scope = {k: r for k, r in generated.items() if not r["in_scope"]}   # should be refused
answered = {k: r for k, r in in_scope.items() if not r["refused"]}

## Safety and trust metrics

In [7]:
refusal_rate = sum(r["refused"] for r in out_scope.values()) / max(1, len(out_scope))
false_refusal_rate = sum(r["refused"] for r in in_scope.values()) / max(1, len(in_scope))
citation_correctness = sum(r["gold_page"] in r["cited_pages"] for r in answered.values()) / max(1, len(answered))

print(f"Refusal rate (out-of-scope):   {refusal_rate:.3f}")
print(f"False-refusal rate (in-scope): {false_refusal_rate:.3f}")
print(f"Citation correctness:          {citation_correctness:.3f}")
print("Out-of-scope NOT refused:", [k for k, r in out_scope.items() if not r["refused"]])
print("In-scope wrongly refused:", [k for k, r in in_scope.items() if r["refused"]])

Refusal rate (out-of-scope):   1.000
False-refusal rate (in-scope): 0.000
Citation correctness:          0.500
Out-of-scope NOT refused: []
In-scope wrongly refused: []


## Answer quality - ROUGE-1 and BERTScore

In [8]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score

ids = list(answered)
bot = [answered[k]["answer"] for k in ids]
ideal = [answered[k]["reference"] for k in ids]

scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)
rouge_scores = [scorer.score(i, b)["rouge1"].fmeasure for i, b in zip(ideal, bot)]
bert_scores = bert_score(bot, ideal, model_type="distilbert-base-uncased", lang="en")[2].tolist()

rouge_average = sum(rouge_scores) / len(rouge_scores)
bert_average = sum(bert_scores) / len(bert_scores)

pd.DataFrame({"id": ids, "rouge1_f1": rouge_scores, "bertscore_f1": bert_scores,
              "answer": bot, "ideal_answer": ideal}).to_csv(f"{RESULTS_DIR}/answer_quality_{RUN_NAME}.csv", index=False)

print(f"Questions scored: {len(ids)}")
print(f"ROUGE-1 F1:   {rouge_average:.3f}")
print(f"BERTScore F1: {bert_average:.3f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Questions scored: 4
ROUGE-1 F1:   0.477
BERTScore F1: 0.836


## DKT answer accuracy

In [9]:
from sentence_transformers import SentenceTransformer, util

# The bot writes free text, so a DKT answer counts as correct if it contains
# most of the key words of the correct option, or has a very close meaning.
skip = set("a an the and or but if of to in on at by for with from as is are was were be been it its this that you your must should can may will not no yes do does when what which".split())
clean = lambda text: re.sub(r"\W+", " ", text.lower()).strip()
model = SentenceTransformer("all-MiniLM-L6-v2")

rows = []
for k, r in in_scope.items():
    if not r["dkt_answer"]:
        continue
    overlap = similarity = 0.0                       # a refused answer stays at 0 = wrong
    if not r["refused"]:
        key_words = [w for w in clean(r["dkt_answer"]).split() if w not in skip]
        overlap = sum(w in clean(r["answer"]).split() for w in key_words) / max(1, len(key_words))
        similarity = float(util.cos_sim(model.encode(r["answer"]), model.encode(r["dkt_answer"])))
    rows.append({"id": k, "question": r["question"], "correct_dkt_answer": r["dkt_answer"],
                 "bot_answer": r["answer"], "word_overlap": round(overlap, 2),
                 "similarity": round(similarity, 2),
                 "counted_as_correct": overlap >= THRESHOLD or similarity >= THRESHOLD})

dkt_table = pd.DataFrame(rows)
dkt_table.to_csv(f"{RESULTS_DIR}/dkt_check_{RUN_NAME}.csv", index=False)
dkt_accuracy = dkt_table["counted_as_correct"].mean()
print(f"DKT answer accuracy (proxy): {dkt_accuracy:.3f}")
print("Spot-check about 10 rows of dkt_check_" + RUN_NAME + ".csv by hand.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

DKT answer accuracy (proxy): 0.250
Spot-check about 10 rows of dkt_check_baseline.csv by hand.


## Faithfulness sheet

In [10]:
random.seed(42)
sample = random.sample(list(answered), min(20, len(answered)))

pd.DataFrame([{"id": k, "question": generated[k]["question"], "answer": generated[k]["answer"],
               "retrieved_chunks": "\n---\n".join(generated[k]["retrieved_text"]),
               "supported? (yes / partly / no)": "", "what was added or mixed up": ""}
              for k in sample]).to_csv(f"{RESULTS_DIR}/faithfulness_sheet_{RUN_NAME}.csv", index=False)

print(f"Saved faithfulness_sheet_{RUN_NAME}.csv ({len(sample)} answers). Open it in Excel and fill in the last 2 columns.")

Saved faithfulness_sheet_baseline.csv (4 answers). Open it in Excel and fill in the last 2 columns.


## Summary table

In [11]:
summary = pd.Series({
    "ROUGE-1 F1": rouge_average,
    "BERTScore F1": bert_average,
    "Refusal rate (out-of-scope)": refusal_rate,
    "False-refusal rate (in-scope)": false_refusal_rate,
    "Citation correctness": citation_correctness,
    "DKT answer accuracy (proxy)": dkt_accuracy,
    "Average seconds per answer": sum(r["seconds"] for r in generated.values()) / len(generated),
}).round(3)

summary.to_csv(f"{RESULTS_DIR}/summary_{RUN_NAME}.csv", header=["value"])
print(summary.to_string())
print("\nFiles saved in:", RESULTS_DIR)

ROUGE-1 F1                       0.477
BERTScore F1                     0.836
Refusal rate (out-of-scope)      1.000
False-refusal rate (in-scope)    0.000
Citation correctness             0.500
DKT answer accuracy (proxy)      0.250
Average seconds per answer       3.533

Files saved in: /Users/vineetbangera/road_rules_results
